# TetheredAI MLB Modeling Lab — Batted-Ball Features, Correlations, and Interactions

This notebook is designed for your upgraded feature file after adding team batted-ball velocity and batted-ball distance features upstream.

It covers:

- Exact date range and target counts
- Batted-ball feature availability checks
- Missingness rules and missing indicators
- Correlation-to-target and feature-redundancy analysis
- Baseball-logical interaction terms
- Chronological model comparison
- Calibration diagnostics
- Champion model export

Primary model-selection metrics should be **log loss**, **Brier score**, and **calibration**, not accuracy alone.


In [ ]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score, accuracy_score, classification_report, confusion_matrix
from sklearn.inspection import permutation_importance

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

FEATURES_PATH = PROJECT_ROOT / 'data' / 'processed' / 'mlb_game_features.parquet'
MODEL_DIR = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('FEATURES_PATH exists:', FEATURES_PATH.exists(), FEATURES_PATH)


## 1. Load feature file and verify date range

The exact pull range is determined by the feature file, not by memory of the Cloud Run settings. This section shows exactly what is in the file you are training from.


In [ ]:
features = pd.read_parquet(FEATURES_PATH)

if 'official_date' in features.columns:
    features['official_date'] = pd.to_datetime(features['official_date'], errors='coerce')
if 'game_datetime_utc' in features.columns:
    features['game_datetime_utc'] = pd.to_datetime(features['game_datetime_utc'], utc=True, errors='coerce')

print('Rows:', len(features))
print('Columns:', len(features.columns))
print('All official_date range:', features['official_date'].min(), 'to', features['official_date'].max())

completed = features[features['target_home_win'].notna()].copy()
upcoming = features[features['target_home_win'].isna()].copy()
completed['target_home_win'] = completed['target_home_win'].astype(int)

print('Completed/labeled rows:', len(completed))
print('Upcoming/unfinal rows:', len(upcoming))
print('Completed date range:', completed['official_date'].min(), 'to', completed['official_date'].max())
print('Upcoming/unfinal date range:', upcoming['official_date'].min(), 'to', upcoming['official_date'].max() if len(upcoming) else None)
print('
Target counts:')
display(features['target_home_win'].value_counts(dropna=False).to_frame('count'))

home_rate = completed['target_home_win'].mean()
print('Home win rate baseline:', round(home_rate, 4))


## 2. Feature discovery

This builds feature lists by family. It excludes IDs, dates, raw names, and targets. It also identifies the new batted-ball velocity/distance columns created upstream.


In [ ]:
TARGET_COL = 'target_home_win'

EXCLUDE_EXACT = {
    'target_home_win', 'home_score', 'away_score', 'home_team_name', 'away_team_name',
    'official_date', 'game_datetime_utc', 'run_id', 'scored_at_utc',
}
EXCLUDE_CONTAINS = [
    'name', 'desc', 'description', 'url', 'id_sid',
]
ID_COL_SUFFIXES = ('_id', '_pk')

def is_candidate_feature(col: str) -> bool:
    if col in EXCLUDE_EXACT:
        return False
    low = col.lower()
    if any(x in low for x in EXCLUDE_CONTAINS):
        return False
    if col.endswith(ID_COL_SUFFIXES) or col in {'game_pk'}:
        return False
    if col.startswith('target_'):
        return False
    return pd.api.types.is_numeric_dtype(features[col])

feature_cols_all = [c for c in features.columns if is_candidate_feature(c)]

statcast_cols = [c for c in feature_cols_all if ('statcast' in c.lower()) or ('_sc_' in c.lower()) or c.startswith(('home_sc_', 'away_sc_', 'diff_sc_')) or 'batted_ball' in c.lower()]
batted_ball_cols = [c for c in feature_cols_all if 'batted_ball' in c.lower() or 'avg_ev' in c.lower() or 'max_ev' in c.lower() or 'p90_batted' in c.lower() or 'distance' in c.lower()]
market_cols = [c for c in feature_cols_all if c.startswith('market_') or 'moneyline' in c.lower() or 'implied_prob' in c.lower()]
elo_cols = [c for c in feature_cols_all if 'elo' in c.lower()]
starter_cols = [c for c in feature_cols_all if 'starter' in c.lower()]
bullpen_cols = [c for c in feature_cols_all if 'bullpen' in c.lower()]
team_cols = [c for c in feature_cols_all if ('team_' in c.lower() or c.startswith(('home_', 'away_', 'diff_'))) and c not in starter_cols + bullpen_cols]

print('All candidate numeric features:', len(feature_cols_all))
print('Statcast features:', len(statcast_cols))
print('Batted-ball features:', len(batted_ball_cols))
print('Market features:', len(market_cols))
print('Elo features:', len(elo_cols))
print('Starter features:', len(starter_cols))
print('Bullpen features:', len(bullpen_cols))

print('
Batted-ball feature sample:')
display(pd.Series(batted_ball_cols, name='batted_ball_cols').head(100).to_frame())


## 3. Missingness rules

Rules used here:

- Drop very-high-missing features above `DROP_IF_MISSING_GT` unless you manually keep them.
- Drop stolen-base success-rate features because they are denominator-driven and unstable.
- Add missing indicators automatically inside `SimpleImputer(add_indicator=True)`.

This means starter Statcast features with around 10% missingness are kept.


In [ ]:
DROP_IF_MISSING_GT = 0.35
MANUAL_DROP_CONTAINS = [
    'sb_success_rate',
]

missing_pct = completed[feature_cols_all].isna().mean().sort_values(ascending=False)
missing_summary = missing_pct.to_frame('missing_pct')
display(missing_summary.head(80))

drop_high_missing = missing_pct[missing_pct > DROP_IF_MISSING_GT].index.tolist()
manual_drop = [c for c in feature_cols_all if any(x in c.lower() for x in MANUAL_DROP_CONTAINS)]

# Keep market columns out of pure-baseball modeling unless explicitly testing market-aware models.
# They may be all-missing historically if odds were not archived.
drop_cols_base = sorted(set(drop_high_missing + manual_drop))

feature_cols_clean = [c for c in feature_cols_all if c not in drop_cols_base]

print('Dropped columns:', len(drop_cols_base))
display(pd.Series(drop_cols_base, name='dropped').to_frame().head(120))
print('Remaining features:', len(feature_cols_clean))


## 4. Correlation diagnostics

Correlation is not the final model judge, but it helps identify:

- Redundant features
- Broken/leaky features
- Candidate interaction terms
- Signals that are directionally useful


In [ ]:
numeric = completed[feature_cols_clean].select_dtypes(include='number').copy()

corr_to_target = (
    numeric.assign(target_home_win=completed[TARGET_COL])
    .corr(numeric_only=True)[TARGET_COL]
    .drop(TARGET_COL)
    .dropna()
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

print('Top absolute correlations to target:')
display(corr_to_target.head(60).to_frame('corr_to_home_win'))

print('Most negative correlations to target:')
display(corr_to_target.sort_values().head(40).to_frame('corr_to_home_win'))

print('Most positive correlations to target:')
display(corr_to_target.sort_values(ascending=False).head(40).to_frame('corr_to_home_win'))


In [ ]:
# Highly correlated feature pairs among a manageable subset.
TOP_N_FOR_REDUNDANCY = 120
top_features_for_corr = corr_to_target.abs().head(TOP_N_FOR_REDUNDANCY).index.tolist()

corr_abs = numeric[top_features_for_corr].corr(numeric_only=True).abs()
upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))

high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={'level_0': 'feature_1', 'level_1': 'feature_2', 0: 'abs_corr'})
    .sort_values('abs_corr', ascending=False)
)

display(high_corr_pairs.head(100))


In [ ]:
# Correlation heatmap for the top target-correlated features.
heatmap_features = corr_to_target.abs().head(30).index.tolist()

if heatmap_features:
    mat = numeric[heatmap_features].corr(numeric_only=True)
    plt.figure(figsize=(12, 10))
    plt.imshow(mat, aspect='auto')
    plt.xticks(range(len(heatmap_features)), heatmap_features, rotation=90)
    plt.yticks(range(len(heatmap_features)), heatmap_features)
    plt.colorbar()
    plt.title('Correlation among top target-correlated features')
    plt.tight_layout()
    plt.show()


## 5. Batted-ball comparison and interaction features

The upstream patches should create columns such as:

- `diff_team_off_sc_avg_batted_ball_ev_last20`
- `diff_team_off_sc_avg_batted_ball_distance_last20`
- `diff_team_vs_hand_sc_avg_batted_ball_ev_last20`
- `diff_team_vs_hand_sc_avg_batted_ball_distance_last20`

This section also creates a small set of baseball-logical interactions for testing. Do not blindly keep all interactions; promote them only when walk-forward log loss/Brier improve.


In [ ]:
def add_interaction_terms(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    interaction_specs = [
        # Opponent contact quality × opposing starter contact allowed
        ('diff_team_off_sc_avg_batted_ball_ev_last20', 'diff_starter_statcast_sc_avg_batted_ball_ev_allowed_last10', 'int_team_ev_x_starter_ev_allowed'),
        ('diff_team_off_sc_avg_batted_ball_distance_last20', 'diff_starter_statcast_sc_avg_batted_ball_distance_allowed_last10', 'int_team_dist_x_starter_dist_allowed'),
        ('diff_team_off_sc_barrel_rate_last20', 'diff_starter_statcast_sc_barrel_rate_allowed_last10', 'int_team_barrel_x_starter_barrel_allowed'),
        ('diff_team_off_sc_hard_hit_rate_last20', 'diff_starter_statcast_sc_hard_hit_rate_allowed_last10', 'int_team_hardhit_x_starter_hardhit_allowed'),

        # Team offense vs hand × starter quality
        ('diff_team_vs_hand_sc_avg_batted_ball_ev_last20', 'diff_starter_statcast_sc_xwoba_allowed_contact_last10', 'int_vs_hand_ev_x_starter_xwoba_allowed'),
        ('diff_team_vs_hand_sc_barrel_rate_last20', 'diff_starter_statcast_sc_barrel_rate_allowed_last10', 'int_vs_hand_barrel_x_starter_barrel_allowed'),

        # Bullpen fatigue/quality × opponent offense quality
        ('diff_sc_bullpen_pitches_last3', 'diff_team_off_sc_hard_hit_rate_last20', 'int_bullpen_pitches_x_team_hardhit'),
        ('diff_bullpen_sc_xwoba_allowed_contact_last5', 'diff_team_off_sc_xwoba_contact_last20', 'int_bullpen_xwoba_allowed_x_team_xwoba'),
    ]

    created = []
    for a, b, name in interaction_specs:
        if a in out.columns and b in out.columns:
            out[name] = pd.to_numeric(out[a], errors='coerce') * pd.to_numeric(out[b], errors='coerce')
            created.append(name)

    print('Created interaction terms:', len(created))
    if created:
        display(pd.Series(created, name='interaction_features').to_frame())
    return out

completed_i = add_interaction_terms(completed)
features_i = add_interaction_terms(features)

interaction_cols = [c for c in completed_i.columns if c.startswith('int_') and pd.api.types.is_numeric_dtype(completed_i[c])]
feature_cols_with_interactions = [c for c in feature_cols_clean if c in completed_i.columns] + interaction_cols

print('Feature count with interactions:', len(feature_cols_with_interactions))


## 6. Chronological split

Never use random splits for this problem. We use a chronological holdout so the model is tested on future-like data.


In [ ]:
def chronological_split(df: pd.DataFrame, test_frac: float = 0.20):
    df = df.sort_values(['official_date', 'game_datetime_utc', 'game_pk']).reset_index(drop=True)
    split_idx = int(len(df) * (1 - test_frac))
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

train_df, test_df = chronological_split(completed_i, test_frac=0.20)

print('Train rows:', len(train_df), train_df['official_date'].min(), 'to', train_df['official_date'].max())
print('Test rows:', len(test_df), test_df['official_date'].min(), 'to', test_df['official_date'].max())

y_train = train_df[TARGET_COL].astype(int)
y_test = test_df[TARGET_COL].astype(int)

baseline_train_rate = y_train.mean()
baseline_probs = np.repeat(baseline_train_rate, len(y_test))

baseline_metrics = {
    'model_name': 'constant_train_home_rate',
    'n_test': len(y_test),
    'avg_pred': float(baseline_probs.mean()),
    'actual_rate': float(y_test.mean()),
    'log_loss': float(log_loss(y_test, baseline_probs)),
    'brier': float(brier_score_loss(y_test, baseline_probs)),
    'roc_auc': 0.5,
    'accuracy_50pct': float(accuracy_score(y_test, baseline_probs >= 0.5)),
}

display(pd.DataFrame([baseline_metrics]))


## 7. Model feature sets

The goal is to identify whether batted-ball features and interaction terms actually improve the proper scoring metrics.


In [ ]:
def cols_matching(cols, include_any=(), exclude_any=()):
    out = []
    for c in cols:
        low = c.lower()
        if include_any and not any(x in low for x in include_any):
            continue
        if exclude_any and any(x in low for x in exclude_any):
            continue
        out.append(c)
    return out

# Pure baseball features exclude market columns by default.
pure_base_cols = [c for c in feature_cols_clean if c not in market_cols]
pure_base_cols_i = [c for c in feature_cols_with_interactions if c not in market_cols]

feature_sets = {
    'baseline_no_market': pure_base_cols,
    'batted_ball_only': cols_matching(pure_base_cols_i, include_any=('batted_ball', 'avg_ev', 'max_ev', 'p90_batted', 'distance', 'hard_hit', 'barrel')),
    'statcast_only': [c for c in pure_base_cols_i if c in statcast_cols or c in interaction_cols],
    'starter_statcast': cols_matching(pure_base_cols_i, include_any=('starter_statcast',)),
    'team_statcast': cols_matching(pure_base_cols_i, include_any=('team_off_sc', 'team_vs_hand_sc', 'batted_ball')),
    'enhanced_with_interactions': pure_base_cols_i,
}

# Optional market-aware set once you have adequate historical odds coverage.
market_available = [c for c in market_cols if completed_i[c].notna().mean() > 0.50] if market_cols else []
if market_available:
    feature_sets['market_aware'] = pure_base_cols_i + market_available

for name, cols in feature_sets.items():
    cols = [c for c in cols if c in completed_i.columns and pd.api.types.is_numeric_dtype(completed_i[c])]
    feature_sets[name] = cols
    print(name, len(cols))


## 8. Define models

This includes a logit benchmark and several tree/boosting models. XGBoost and LightGBM are optional.


In [ ]:
def make_pipeline(model, scale=False):
    steps = [('imputer', SimpleImputer(strategy='median', add_indicator=True))]
    if scale:
        steps.append(('scaler', StandardScaler()))
    steps.append(('model', model))
    return Pipeline(steps)

models = {
    'logit_l2': make_pipeline(LogisticRegression(max_iter=5000, C=0.5, solver='lbfgs'), scale=True),
    'random_forest': make_pipeline(RandomForestClassifier(n_estimators=500, min_samples_leaf=25, max_features='sqrt', random_state=42, n_jobs=-1)),
    'extra_trees': make_pipeline(ExtraTreesClassifier(n_estimators=500, min_samples_leaf=25, max_features='sqrt', random_state=42, n_jobs=-1)),
    'hist_gbdt': make_pipeline(HistGradientBoostingClassifier(max_iter=300, learning_rate=0.03, max_leaf_nodes=15, l2_regularization=0.1, random_state=42)),
}

try:
    from xgboost import XGBClassifier
    models['xgboost'] = make_pipeline(XGBClassifier(
        n_estimators=300,
        max_depth=2,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.9,
        min_child_weight=10,
        reg_alpha=0.0,
        reg_lambda=2.0,
        eval_metric='logloss',
        random_state=42,
        n_jobs=-1,
    ))
except Exception as e:
    print('Skipping XGBoost:', repr(e))

try:
    from lightgbm import LGBMClassifier
    models['lightgbm'] = make_pipeline(LGBMClassifier(
        n_estimators=400,
        max_depth=3,
        learning_rate=0.03,
        num_leaves=15,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_alpha=0.0,
        reg_lambda=2.0,
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    ))
except Exception as e:
    print('Skipping LightGBM:', repr(e))

print(models.keys())


## 9. Fit/evaluate model × feature set grid

This keeps runtime reasonable by using a single chronological holdout. After identifying promising configurations, you can add walk-forward folds.


In [ ]:
def evaluate_model(name, model, cols):
    X_train = train_df[cols].copy()
    X_test = test_df[cols].copy()

    fitted = clone(model)
    fitted.fit(X_train, y_train)

    if hasattr(fitted, 'predict_proba'):
        p = fitted.predict_proba(X_test)[:, 1]
    else:
        # Very rare fallback.
        raw = fitted.decision_function(X_test)
        p = 1 / (1 + np.exp(-raw))

    p = np.clip(p, 1e-6, 1 - 1e-6)
    pred = (p >= 0.5).astype(int)

    metrics = {
        'model_name': name,
        'feature_count': len(cols),
        'n_test': len(y_test),
        'avg_pred': float(np.mean(p)),
        'actual_rate': float(y_test.mean()),
        'log_loss': float(log_loss(y_test, p)),
        'brier': float(brier_score_loss(y_test, p)),
        'roc_auc': float(roc_auc_score(y_test, p)),
        'accuracy_50pct': float(accuracy_score(y_test, pred)),
    }
    return fitted, p, metrics

results = []
fitted_models = {}
preds_by_name = {}

for fs_name, cols in feature_sets.items():
    if len(cols) < 3:
        print('Skipping tiny feature set:', fs_name, len(cols))
        continue
    for model_name, model in models.items():
        full_name = f'{fs_name}__{model_name}'
        try:
            fitted, p, metrics = evaluate_model(full_name, model, cols)
            results.append(metrics)
            fitted_models[full_name] = (fitted, cols)
            preds_by_name[full_name] = p
            print(full_name, 'log_loss=', round(metrics['log_loss'], 4), 'brier=', round(metrics['brier'], 4), 'auc=', round(metrics['roc_auc'], 4))
        except Exception as e:
            print('FAILED', full_name, repr(e))

results_df = pd.DataFrame(results).sort_values(['log_loss', 'brier']).reset_index(drop=True)
display(results_df)


## 10. Champion diagnostics


In [ ]:
best_name = results_df.iloc[0]['model_name']
best_model, best_cols = fitted_models[best_name]
best_p = preds_by_name[best_name]

print('Best model:', best_name)
print('Feature count:', len(best_cols))
print(results_df.iloc[0].to_dict())

print('
Classification report at 50% threshold:')
print(classification_report(y_test, (best_p >= 0.5).astype(int), digits=3))
print('Confusion matrix:')
display(pd.DataFrame(confusion_matrix(y_test, (best_p >= 0.5).astype(int)), index=['actual_away','actual_home'], columns=['pred_away','pred_home']))

calib = pd.DataFrame({'p': best_p, 'y': y_test.values})
calib['bucket'] = pd.cut(calib['p'], bins=[0, .35, .40, .45, .50, .55, .60, .65, .70, 1.0], include_lowest=True)
calib_table = calib.groupby('bucket', observed=False).agg(
    games=('y','size'),
    avg_pred_prob=('p','mean'),
    actual_home_win_rate=('y','mean'),
)
calib_table['calibration_error'] = calib_table['actual_home_win_rate'] - calib_table['avg_pred_prob']
display(calib_table)

plt.figure(figsize=(7,5))
plt.plot(calib_table['avg_pred_prob'], calib_table['actual_home_win_rate'], marker='o')
plt.plot([0,1], [0,1], linestyle='--')
plt.xlabel('Average predicted probability')
plt.ylabel('Actual home win rate')
plt.title(f'Calibration: {best_name}')
plt.tight_layout()
plt.show()


## 11. Permutation importance

This is slower, so it uses the holdout set and the current champion only.


In [ ]:
RUN_PERMUTATION_IMPORTANCE = True

if RUN_PERMUTATION_IMPORTANCE:
    X_test_best = test_df[best_cols].copy()
    perm = permutation_importance(
        best_model,
        X_test_best,
        y_test,
        scoring='neg_log_loss',
        n_repeats=5,
        random_state=42,
        n_jobs=-1,
    )
    imp = pd.DataFrame({
        'feature': best_cols,
        'importance_mean': perm.importances_mean,
        'importance_std': perm.importances_std,
    }).sort_values('importance_mean', ascending=False)
    display(imp.head(80))
else:
    print('Permutation importance skipped.')


## 12. Optional champion export

Set `APPROVE_EXPORT = True` only after you are comfortable with the diagnostics. The bundle includes the model and exact feature columns, which keeps production scoring consistent.


In [ ]:
APPROVE_EXPORT = False

if APPROVE_EXPORT:
    import joblib
    bundle = {
        'model': best_model,
        'feature_cols': best_cols,
        'model_name': best_name,
        'metrics': results_df.iloc[0].to_dict(),
        'target_col': TARGET_COL,
        'created_from_features_path': str(FEATURES_PATH),
    }
    model_path = MODEL_DIR / 'mlb_moneyline_champion.joblib'
    metadata_path = MODEL_DIR / 'mlb_moneyline_champion_metadata.json'
    joblib.dump(bundle, model_path)
    metadata_path.write_text(json.dumps({k: v for k, v in bundle.items() if k != 'model'}, default=str, indent=2))
    print('Saved:', model_path)
    print('Saved:', metadata_path)
else:
    print('APPROVE_EXPORT is False; not exporting.')
